# Master Exam Template: Clustering Models
## Course Scope: K-Means (Elbow & Silhouette) and Agglomerative Hierarchical Clustering (Dendrogram)
**Matches:** 7CS353 Lab Assignment 9  
**Marks Rubric (40 Marks):**
* Section 1: EDA [10 Marks]
* Section 2: Implementation (StandardScaler & Baseline K-Means) [10 Marks]
* Section 3: Evaluation (Elbow Method Inertia, Silhouette Score, Cluster Scatter) [10 Marks]
* Section 4: Modify Technique (Hierarchical Clustering with Ward Dendrogram) [10 Marks]

In [ ]:
# 1. Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import dendrogram, linkage

plt.rcParams['figure.figsize'] = (8, 4)
print("Libraries imported!")

In [ ]:
# Section 1: EDA (10 Marks) - Quick Exam Dataset Generation
np.random.seed(42)
n = 300

f1 = np.concatenate([np.random.normal(20, 4, 100), np.random.normal(50, 5, 100), np.random.normal(80, 6, 100)])
f2 = np.concatenate([np.random.normal(70, 5, 100), np.random.normal(30, 4, 100), np.random.normal(60, 5, 100)])
f3 = np.random.uniform(10, 50, n)

df = pd.DataFrame({'Metric_A': f1, 'Metric_B': f2, 'Metric_C': f3})

print("Dataset shape:", df.shape)
print("Missing values:\n", df.isnull().sum())
display(df.describe().round(2))

plt.figure(figsize=(6, 3))
sns.scatterplot(x='Metric_A', y='Metric_B', data=df, color='navy', alpha=0.7)
plt.title("Scatter Plot: Metric_A vs Metric_B")
plt.show()

In [ ]:
# Section 2: Implementation (10 Marks) - Scaling & K-Means (k=3)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df['cluster_kmeans'] = kmeans.fit_predict(X_scaled)

print("K-Means (k=3) fitted!")
print("Cluster counts:\n", df['cluster_kmeans'].value_counts())

In [ ]:
# Section 3: Evaluation (10 Marks) - Elbow Curve, Silhouette & Cluster Plot
k_values = range(2, 7)
inertias = []
sil_scores = []

for k in k_values:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(k_values, inertias, marker='o', color='red')
axes[0].set_title("Elbow Method (Inertia vs k)")
axes[1].plot(k_values, sil_scores, marker='s', color='teal')
axes[1].set_title("Silhouette Score vs k")
plt.tight_layout()
plt.show()

print(f"K-Means (k=3) Silhouette Score: {silhouette_score(X_scaled, df['cluster_kmeans']):.4f}")

plt.figure(figsize=(6, 3))
sns.scatterplot(x='Metric_A', y='Metric_B', hue='cluster_kmeans', data=df, palette='Set1')
plt.title("K-Means Cluster Partitioning")
plt.show()

In [ ]:
# Section 4: Modify Technique (10 Marks) - Hierarchical Clustering & Dendrogram
linkage_data = linkage(X_scaled, method='ward')

plt.figure(figsize=(8, 3))
dendrogram(linkage_data, truncate_mode='lastp', p=12)
plt.title("Hierarchical Dendrogram (Ward Linkage)")
plt.axhline(12, color='red', linestyle='--', label='Cut-off line (k=3)')
plt.legend()
plt.show()

agg = AgglomerativeClustering(n_clusters=3, linkage='ward')
df['cluster_hierarchical'] = agg.fit_predict(X_scaled)

comparison_df = pd.DataFrame([
    {'Algorithm': 'K-Means (k=3)', 'Silhouette Score': round(silhouette_score(X_scaled, df['cluster_kmeans']), 4)},
    {'Algorithm': 'Hierarchical Agglomerative (Ward)', 'Silhouette Score': round(silhouette_score(X_scaled, df['cluster_hierarchical']), 4)}
])
print("=== Final Clustering Comparison Table ===")
display(comparison_df)